# Reference copy - needs non-public data

이 노트북은 원본 연구 노트북의 **참고용 사본**입니다. 실행에 필요한 전처리 결과 `Research_Data_Final.csv`(01 노트북 출력, 유료 데이터 기반)은(는) 유료/비공개 데이터라 이 저장소에 포함하지 않았으므로 **그대로는 끝까지 실행되지 않습니다.** 출력, 실행 카운트, 로컬 경로는 제거했습니다. 실행 가능한 데모는 `00_public_sample_workflow.ipynb`(합성 데이터)를 사용하세요.

In [ ]:
import copy
import warnings
warnings.filterwarnings('ignore')

import re
import pandas as pd
import numpy as np
from tqdm import tqdm
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


from scipy.stats.mstats import winsorize

## **데이터 불러오기**

In [ ]:
# [분석/모델링 노트북의 첫 셀]
import pandas as pd
import numpy as np

# 2. 데이터 불러오기
# - index_col=[0, 1]: 첫 두 컬럼(Date, Ticker)을 MultiIndex로 지정
# - parse_dates=[0]: 첫 번째 컬럼(Date)을 날짜 형식으로 인식
data = pd.read_csv("Research_Data_Final.csv", index_col=[0, 1], parse_dates=[0], encoding="CP949")

print(f"Data Loaded. Shape: {data.shape}")
data.head()

### **데이터 클렌징(가격 관련 변수 제거)**

In [ ]:
# [Code Cell]
# 제거할 변수 목록 (필요시 주석 처리하여 다시 살릴 수 있음)
price_vars = [
    'M_B', 
    'Beta', 
    'Log_MarketCap'
]

# 해당 변수 제거 (Drop)
clean_data = data# .drop(columns=price_vars, errors='ignore')

# 최종 데이터 확인
print("=== Final Dataset for Analysis ===")
print("Columns:", clean_data.columns.tolist())
print("-" * 30)
print(clean_data.describe().T[['count', 'mean', 'std']])

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. Extreme Outlier Clipping (경제적 상식 범위로 제한)
# 이유: 표준편차가 수백만이 나오면 OLS가 학습을 못함.
clean_data['Interest_Cov'] = clean_data['Interest_Cov'].clip(-100, 100)  # 이자보상배율 ±100배면 충분함
clean_data['Debt_Ratio'] = clean_data['Debt_Ratio'].clip(0, 1000)        # 부채비율 1000% 이상은 그냥 '위험'으로 통일
clean_data['Payout_Ratio'] = clean_data['Payout_Ratio'].clip(0, 200)     # 배당성향 200% 이상은 비정상
clean_data['ROE'] = clean_data['ROE'].clip(-100, 100)                    # ROE ±100% 제한

# 다시 통계량 확인
print("=== Re-checked Statistics (After Clipping) ===")
print(clean_data[['Interest_Cov', 'Debt_Ratio', 'Payout_Ratio', 'ROE']].describe().T)

# 2. VIF (다중공선성) 체크
# Target(EP)을 제외한 모든 X변수 선택
features = clean_data.columns.drop(['EP'])
X = clean_data[features]
X = sm.add_constant(X) # 상수항 추가

# VIF 계산
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

print("\n=== VIF Check (Multicollinearity) ===")
print(vif_data.sort_values(by='VIF', ascending=False))

## **Full-Sample Model Fitting (Benchmark vs Challenger)**

In [ ]:
# 3. Full-Sample OLS 수행
y = clean_data['EP']

model_ols = sm.OLS(y, X).fit()

print("\n=== Full-Sample OLS Results ===")
print(model_ols.summary())

# 결과 저장
clean_data['Expected_EP_OLS'] = model_ols.fittedvalues
clean_data['Mispricing_OLS'] = clean_data['Expected_EP_OLS'] - clean_data['EP']

# model_ols.summary()

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 4. XGBoost 수행
# Hyperparameters: 일단 기본값 근처로 시작. 나중에 튜닝 가능.
xgb_model = XGBRegressor(
    n_estimators=1000,       # 트리 개수
    learning_rate=0.05,      # 학습률
    max_depth=6,             # 트리 깊이
    subsample=0.8,           # 데이터 샘플링 비율
    colsample_bytree=0.8,    # 컬럼 샘플링 비율
    n_jobs=-1,               # CPU 병렬처리
    random_state=42,
    device='cuda', tree_method='hist' # GPU 사용 시 주석 해제
)

print("\nTraining XGBoost...")
xgb_model.fit(X.drop(columns=['const']), y) # XGB는 상수항(const) 불필요

# 결과 저장
clean_data['Expected_EP_XGB'] = xgb_model.predict(X.drop(columns=['const']))
clean_data['Mispricing_XGB'] = clean_data['Expected_EP_XGB'] - clean_data['EP']

# 5. 간단 성능 비교 (In-Sample)
r2_ols = model_ols.rsquared
r2_xgb = r2_score(y, clean_data['Expected_EP_XGB'])

print(f"\n[In-Sample Performance Check]")
print(f"OLS R-squared:    {r2_ols:.4f}")
print(f"XGBoost R-squared: {r2_xgb:.4f}")

if r2_xgb > r2_ols:
    print(">> XGBoost가 데이터를 더 잘 설명하고 있습니다. (Good)")
else:
    print(">> Warning: XGBoost 성능이 OLS보다 낮습니다. 파라미터 튜닝이 필요합니다.")

In [ ]:
# 현재 clean_data에 어떤 컬럼들이 있는지 확인
print("Current Columns:", clean_data.columns.tolist())

# [중요] Full-Sample 분석 때 생성된 파생변수들을 싹 다 지워야 함
leakage_vars = [
    'Expected_EP_OLS', 'Mispricing_OLS', 
    'Expected_EP_XGB', 'Mispricing_XGB'
]

# 해당 컬럼이 있다면 제거
clean_data = clean_data.drop(columns=leakage_vars, errors='ignore')

print("\n[Fixed] Columns for Rolling Analysis:")
print(clean_data.columns.tolist()) 

---
## **Rolling Window Analysis (OLS: CPU Parallel, XGB: GPU)**

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from xgboost import XGBRegressor
from joblib import Parallel, delayed
from tqdm import tqdm

# =========================================================
# 1. 설정 (Configuration)
# =========================================================
WINDOW_SIZE = 60         # 60개월 롤링 윈도우
dates = clean_data.index.get_level_values(0).unique().sort_values()

# 테스트 기간: 초기 60개월(Window Size) 이후부터 시작
# 즉, 데이터가 60개월 쌓인 61개월 차부터 예측 시작
test_dates = dates[WINDOW_SIZE:]

print(f"Total Test Months: {len(test_dates)}")
print(f"Window Size: {WINDOW_SIZE} months")

---
#### **OLS 전종목**

In [ ]:
# =========================================================
# 2. OLS (CPU Parallel Processing)
# =========================================================
# 핵심 수정: 함수가 (Date, Prediction) 튜플을 반환하도록 변경하여 인덱스 누락 방지
def fit_predict_ols(target_date, all_data, all_dates, window):
    # 1. 날짜 인덱싱
    idx = all_dates.get_loc(target_date)
    start_date = all_dates[idx - window]
    
    # 2. Slicing (메모리 효율보다 안전성을 위해 Boolean Mask 사용)
    # Train: [Start, Target)
    # Test: [Target]
    mask_train = (all_data.index.get_level_values(0) >= start_date) & \
                 (all_data.index.get_level_values(0) < target_date)
    mask_test = (all_data.index.get_level_values(0) == target_date)
    
    train = all_data[mask_train]
    test = all_data[mask_test]
    
    # 데이터 부족 시 건너뜀 (최소 100개 종목은 있어야 회귀분석 의미 있음)
    if len(train) < 100 or len(test) < 10:
        return None
    
    # 3. Fit & Predict
    y_train = train['EP']
    X_train = sm.add_constant(train.drop(columns=['EP']))
    X_test = sm.add_constant(test.drop(columns=['EP']), has_constant='add')
    
    try:
        model = sm.OLS(y_train, X_train).fit()
        pred = model.predict(X_test)
        
        # 결과는 Ticker 인덱스만 가진 Series여야 함
        if isinstance(pred.index, pd.MultiIndex):
            pred = pred.droplevel(0)
            
        return (target_date, pred) # (날짜, 예측결과) 반환
    except:
        return None

print("\n[1/3] Starting OLS (CPU Parallel)...")

# Parallel 실행
ols_raw_results = Parallel(n_jobs=-1)(
    delayed(fit_predict_ols)(date, clean_data, dates, WINDOW_SIZE) 
    for date in tqdm(test_dates)
)

# 결과 정리 (None 제거 및 날짜별 리스트 분리)
ols_valid_dates = []
ols_preds_list = []

for res in ols_raw_results:
    if res is not None:
        ols_valid_dates.append(res[0])
        ols_preds_list.append(res[1])

---
#### **개별종목 OLS**

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from joblib import Parallel, delayed
from tqdm import tqdm

# =========================================================
# 0. 데이터 정렬 (필수)
# =========================================================
# MultiIndex Slicing을 위해 반드시 정렬되어 있어야 합니다.
clean_data.sort_index(level=['Date', 'Ticker'], inplace=True)
print("Data sorted by Date and Ticker.")

# =========================================================
# 1. 설정
# =========================================================
WINDOW_SIZE = 60         
MIN_OBS = 30             

dates = clean_data.index.get_level_values(0).unique().sort_values()
test_dates = dates[WINDOW_SIZE:]

print(f"Total Test Months: {len(test_dates)}")

# =========================================================
# 2. Time-Series OLS 함수 (Robust Version)
# =========================================================
def fit_predict_by_ticker_robust(target_date, all_data, all_dates, window):
    # 1. 기간 설정
    idx = all_dates.get_loc(target_date)
    start_date = all_dates[idx - window]
    
    # 2. Window 데이터 추출 (Date 기준 슬라이싱)
    # (start_date ~ target_date) 구간의 모든 종목 데이터
    # 이 시점에서 window_data의 인덱스는 여전히 (Date, Ticker)
    window_data = all_data.loc[start_date : target_date]
    
    # 3. 예측 대상 종목 선정 (Target Date에 데이터가 있는 종목만)
    try:
        current_slice = window_data.loc[target_date]
        # loc[target_date]를 하면 Date 레벨이 떨어져 나가고 Ticker 인덱스만 남습니다.
        target_tickers = current_slice.index 
    except KeyError:
        return None # 해당 날짜에 데이터가 아예 없는 경우

    predictions = {}
    
    # 4. 종목별 반복 (Loop per Ticker)
    for ticker in target_tickers:
        try:
            # (1) 해당 종목의 시계열 데이터 추출
            # xs를 사용하면 Ticker 레벨을 제거하고 Date 인덱스만 남깁니다.
            # 예: 인덱스가 [2004-03-31, ... , target_date] 가 됨
            ticker_df = window_data.xs(ticker, level='Ticker')
            
            # (2) Train / Test 분리
            # target_date 이전은 Train, 당일은 Test
            mask_train = ticker_df.index < target_date
            
            train = ticker_df[mask_train]
            test_row = ticker_df[~mask_train] # target_date 행
            
            # (3) 최소 관측치 확인
            if len(train) < MIN_OBS:
                continue
                
            # (4) Regression 준비
            y_train = train['EP']
            X_train = train.drop(columns=['EP'])
            X_test = test_row.drop(columns=['EP'])
            
            # 상수항 추가
            X_train = sm.add_constant(X_train)
            # has_constant='add'는 test 데이터에도 상수항(1.0) 컬럼을 강제로 붙임
            X_test = sm.add_constant(X_test, has_constant='add')
            
            # (5) 모델 적합 및 예측
            model = sm.OLS(y_train, X_train).fit()
            pred = model.predict(X_test) 
            
            # 결과 저장 (스칼라 값)
            predictions[ticker] = pred.item() 
            
        except Exception as e:
            # 디버깅을 위해 에러 발생 시 출력 (너무 많으면 주석 처리)
            # print(f"Error on {target_date} / {ticker}: {e}")
            continue

    if not predictions:
        return None
        
    return (target_date, pd.Series(predictions, name='Predicted_EP'))

# =========================================================
# 3. 실행 (검증 후 병렬 처리)
# =========================================================

# [검증 단계] 첫 번째 날짜에 대해 에러가 없는지 먼저 테스트합니다.
print("\n[Test Run] Processing the first target date...")
first_res = fit_predict_by_ticker_robust(test_dates[0], clean_data, dates, WINDOW_SIZE)

if first_res is not None:
    print(f"✅ Success! Sample Predictions for {test_dates[0]}:")
    print(first_res[1].head())
    
    print("\n[Main Run] Starting Parallel Processing for all dates...")
    # 병렬 처리 실행
    ols_raw_results = Parallel(n_jobs=-1)(
        delayed(fit_predict_by_ticker_robust)(date, clean_data, dates, WINDOW_SIZE) 
        for date in tqdm(test_dates)
    )

    # 결과 정리
    ols_valid_dates_ind = []
    ols_preds_list_ind = []

    for res in ols_raw_results:
        if res is not None:
            ols_valid_dates_ind.append(res[0])
            ols_preds_list_ind.append(res[1])
            
    print(f"\nCompleted. Total valid months: {len(ols_valid_dates_ind)}")

else:
    print("❌ Failed on the first date. Please check the error logs.")

---
#### **XGBoost 전종목**

In [ ]:
# =========================================================
# 3. XGBoost (GPU Accelerated)
# =========================================================
print("\n[2/3] Starting XGBoost (GPU)...")

xgb_valid_dates = []
xgb_preds_list = []

# 모델 설정 (GPU 가속)
xgb_model = XGBRegressor(
    n_estimators=500,    # 롤링 윈도우 데이터 규모에 충분
    learning_rate=0.05,
    max_depth=4,         # 과적합 방지
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    device='cuda',       # [핵심] GPU 사용
    tree_method='hist'   # GPU 히스토그램 방식
)

for target_date in tqdm(test_dates):
    # 1. Slicing
    curr_idx = dates.get_loc(target_date)
    start_date = dates[curr_idx - WINDOW_SIZE]
    
    # loc 슬라이싱 (끝점 포함 주의: dates[curr_idx-1]까지)
    train = clean_data.loc[start_date : dates[curr_idx-1]]
    test = clean_data.loc[target_date]
    
    if len(train) < 100 or len(test) < 10: continue
    
    y_train = train['EP']
    X_train = train.drop(columns=['EP'])
    
    # test가 Series일 경우 DataFrame으로 변환
    if isinstance(test, pd.Series):
        test = test.to_frame().T
    X_test = test.drop(columns=['EP'])
    
    # 2. Fit & Predict
    xgb_model.fit(X_train, y_train)
    pred = xgb_model.predict(X_test)
    
    # 결과 저장 (Ticker 인덱스만 있는 Series 생성)
    pred_series = pd.Series(pred, index=X_test.index)
    
    xgb_preds_list.append(pred_series)
    xgb_valid_dates.append(target_date)

---
#### **XGBoost 개별종목**

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from joblib import Parallel, delayed
from tqdm import tqdm

# =========================================================
# 1. 설정 (Configuration)
# =========================================================
WINDOW_SIZE = 60         
MIN_OBS = 30             

# 데이터 정렬 (필수)
clean_data.sort_index(level=['Date', 'Ticker'], inplace=True)
dates = clean_data.index.get_level_values(0).unique().sort_values()
test_dates = dates[WINDOW_SIZE:]

print(f"Total Test Months: {len(test_dates)}")

# =========================================================
# 2. 개별 종목별 XGBoost 함수
# =========================================================
def fit_predict_xgb_by_ticker(target_date, all_data, all_dates, window):
    # 1. 기간 설정 및 데이터 슬라이싱
    idx = all_dates.get_loc(target_date)
    start_date = all_dates[idx - window]
    
    # (start_date ~ target_date) 구간 데이터
    window_data = all_data.loc[start_date : target_date]
    
    # 2. 예측 대상 종목 선정
    try:
        current_slice = window_data.loc[target_date]
        target_tickers = current_slice.index 
    except KeyError:
        return None 

    predictions = {}
    
    # 3. XGBoost 모델 설정 (Small Data 최적화)
    # 60개 데이터에 500개 트리는 너무 많으므로 줄입니다.
    model = XGBRegressor(
        n_estimators=50,       # 500 -> 50 (과적합 방지)
        learning_rate=0.1,     # 0.05 -> 0.1 (트리가 적으므로 학습률 상향)
        max_depth=2,           # 깊이 제한 (복잡도 낮춤)
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=1,              # Joblib 병렬처리를 위해 내부 병렬화는 끔
        random_state=42,
        tree_method='hist'     # CPU에서도 빠름
        # device='cuda'        # 데이터가 작을 땐 CPU가 더 빠름 (오버헤드 방지)
    )
    
    # 4. 종목별 반복
    for ticker in target_tickers:
        try:
            # (1) 데이터 추출
            ticker_df = window_data.xs(ticker, level='Ticker')
            
            mask_train = ticker_df.index < target_date
            train = ticker_df[mask_train]
            test_row = ticker_df[~mask_train] 
            
            # (2) 최소 관측치 확인
            if len(train) < MIN_OBS:
                continue
                
            # (3) X, y 분리
            y_train = train['EP']
            X_train = train.drop(columns=['EP'])
            X_test = test_row.drop(columns=['EP'])
            
            # (4) 학습 및 예측
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            
            predictions[ticker] = pred.item()
            
        except Exception:
            continue

    if not predictions:
        return None
        
    return (target_date, pd.Series(predictions, name='Predicted_EP'))

# =========================================================
# 3. 실행 (CPU Parallel)
# =========================================================
print("\n[2/2] Starting Ticker-wise XGBoost (CPU Parallel)...")

# 병렬 처리 실행 (n_jobs=-1로 모든 CPU 코어 사용)
xgb_raw_results = Parallel(n_jobs=-1)(
    delayed(fit_predict_xgb_by_ticker)(date, clean_data, dates, WINDOW_SIZE) 
    for date in tqdm(test_dates)
)

# 결과 정리
xgb_valid_dates_ind = []
xgb_preds_list_ind = []

for res in xgb_raw_results:
    if res is not None:
        xgb_valid_dates_ind.append(res[0])
        xgb_preds_list_ind.append(res[1])

print(f"Done. Processed {len(xgb_valid_dates_ind)} months.")

---
## **4. 결과 병합 및 저장**


In [ ]:
# =========================================================
# 4. 결과 병합 및 변환 (Merging & Reshaping)
# =========================================================
print("\n[3/3] Reshaping Results...")

# (1) Long Format 생성 (Date Index 복구)
# keys 옵션을 사용하여 [Date, Ticker] MultiIndex 강제 생성 -> 에러 원천 차단
ols_long = pd.concat(ols_preds_list, keys=ols_valid_dates, names=['Date', 'Ticker'])
xgb_long = pd.concat(xgb_preds_list, keys=xgb_valid_dates, names=['Date', 'Ticker'])
ols_long_ind = pd.concat(ols_preds_list_ind, keys=ols_valid_dates_ind, names=['Date', 'Ticker'])
xgb_long_ind = pd.concat(xgb_preds_list_ind, keys=xgb_valid_dates_ind, names=['Date', 'Ticker'])

# (2) Wide Format 변환 (Unstack)
df_ols_wide = ols_long.unstack(level=1)
df_xgb_wide = xgb_long.unstack(level=1)
df_ols_wide_ind = ols_long_ind.unstack(level=1)
df_xgb_wide_ind = xgb_long_ind.unstack(level=1)

# (3) Actual EP Wide Format (괴리 계산용)
# 전체 데이터에서 unstack 후 테스트 기간만 자름
df_actual_wide = clean_data['EP'].unstack(level=1)
df_actual_wide = df_actual_wide.loc[df_actual_wide.index >= test_dates[0]]

# Index를 Datetime으로 확실하게 변환
df_ols_wide.index = pd.to_datetime(df_ols_wide.index)
df_xgb_wide.index = pd.to_datetime(df_xgb_wide.index)  
df_ols_wide_ind.index = pd.to_datetime(df_ols_wide_ind.index)
df_xgb_wide_ind.index = pd.to_datetime(df_xgb_wide_ind.index)

df_actual_wide.index = pd.to_datetime(df_actual_wide.index)

print("-" * 40)
print(f"OLS Result Shape:    {df_ols_wide.shape}")
print(f"XGB Result Shape:    {df_xgb_wide.shape}")
print(f"OLS (Ind) Shape:     {df_ols_wide_ind.shape}")
print(f"XGB (Ind) Shape:     {df_xgb_wide_ind.shape}")
print(f"Actual Data Shape:   {df_actual_wide.shape}")
print("-" * 40)

In [ ]:
# =========================================================
# 5. Mispricing (괴리) 계산
# =========================================================
# 괴리 = 모델이 생각하는 적정 EP - 실제 시장 EP
# (모델EP > 시장EP) => (모델이 보기에 이익수익률이 높아야 함) => (현재 주가가 너무 비쌈? 아니지, 싸다!)
# 잠깐, E/P가 높다 = 주가가 싸다(저평가).
# 모델EP(10%) > 시장EP(5%) -> 모델은 "이 주식 싸게(높은 수익률로) 거래돼야 해"라고 함.
# 실제는 비싸게(낮은 수익률로) 거래됨. -> 즉 "현재 주가는 고평가됨"?
# 아, 부호가 헷갈릴 수 있네.

# [정리]
# High E/P = Undervalued (저평가, Value)
# Low E/P = Overvalued (고평가, Glamour)
# 
# Mispricing = Model_EP - Actual_EP
# 양수(+) => Model_EP가 더 높음 => 모델이 "더 저평가되어야 한다"고 판단 => 즉 "현재 시장가는 너무 비싸다(고평가)?" 가 아니라,
# "모델이 요구하는 수익률이 더 높다"는 뜻.
# 
# 통상적인 Value Signal 정의:
# Signal = Fundamental_Value - Market_Value
# 우리는 Value를 E/P로 쓰고 있음.
# Signal = Expected_EP - Actual_EP
# 이 값이 클수록 "모델이 보는 수익력이 시장가보다 높다" -> 즉 "저평가(Undervalued)" 신호로 해석하는 게 맞네.

# mispricing_ols = df_ols_wide - df_actual_wide
# mispricing_xgb = df_xgb_wide - df_actual_wide

# print("Mispricing Calculated.")

In [ ]:
# # 1. 차원 확인
# print(f"OLS Prediction Shape:    {df_ols_wide.shape}")
# print(f"XGB Prediction Shape:    {df_xgb_wide.shape}")
# print(f"Actual Data Shape:       {df_actual_wide.shape}")

# # 2. 인덱스(날짜) 일치 여부 확인
# dates_match = (df_ols_wide.index.equals(df_actual_wide.index))
# print(f"Dates perfectly match? : {dates_match}")

# # 3. 초기 데이터 확인 (NaN 처리 확인)
# # 가장 첫 번째 테스트 날짜의 데이터 5개만 찍어봄
# first_date = df_ols_wide.index[0]
# print(f"\nFirst Prediction Date: {first_date}")
# print("-" * 30)
# print("OLS Pred:", df_ols_wide.iloc[0, :5].values)
# print("Actual:  ", df_actual_wide.iloc[0, :5].values)
# print("Mispricing:", mispricing_ols.iloc[0, :5].values)

# # 4. 혹시 모를 이상치 체크 (NaN이 아닌데 -무한대나 이상한 값이 있는지)
# print("\nAny +/- Infinity in Mispricing?")
# print(np.isinf(mispricing_ols).sum().sum())

---
## **데이터 저장**

In [ ]:
import numpy as np
import os

# =========================================================
# 5. 차원 동기화 및 괴리율 계산 (Alignment & Calculation)
# =========================================================
print("\n[4/4] Aligning & Calculating Mispricing...")

# 1. 차원 동기화 (Actual Data 기준)
# 예측값 데이터프레임의 인덱스와 컬럼을 실제값에 맞춤 (없는 부분은 NaN 자동 채움)
target_idx, target_cols = df_actual_wide.index, df_actual_wide.columns

df_ols_wide = df_ols_wide.reindex(index=target_idx, columns=target_cols)
df_xgb_wide = df_xgb_wide.reindex(index=target_idx, columns=target_cols)
df_ols_wide_ind = df_ols_wide_ind.reindex(index=target_idx, columns=target_cols)
df_xgb_wide_ind = df_xgb_wide_ind.reindex(index=target_idx, columns=target_cols)

# 2. 괴리율 계산 (Mispricing = Predicted - Actual)
# 양수(+)면 고평가 예측(매도), 음수(-)면 저평가 예측(매수)
mispricing_ols = df_ols_wide - df_actual_wide
mispricing_xgb = df_xgb_wide - df_actual_wide
mispricing_ols_ind = df_ols_wide_ind - df_actual_wide
mispricing_xgb_ind = df_xgb_wide_ind - df_actual_wide

# 3. 데이터 검증 (Shape 확인)
print("-" * 40)
print(f"Actual Data Shape:   {df_actual_wide.shape}")
print(f"Pooled OLS Shape:    {mispricing_ols.shape}")
print(f"Pooled XGB Shape:    {mispricing_xgb.shape}")
print(f"Indiv  OLS Shape:    {mispricing_ols_ind.shape}")
print(f"Indiv  XGB Shape:    {mispricing_xgb_ind.shape}")
print("-" * 40)

# 4. 무결성 체크 (Inf 포함 여부)
inf_check = np.isinf(mispricing_ols).sum().sum() + np.isinf(mispricing_ols_ind).sum().sum()
print(f"Any Infinity values? : {inf_check > 0}")

In [ ]:
# =========================================================
# 6. 결과 저장 (Save Results)
# =========================================================
path = "."
print(f"\nSaving 5 CSV files to '{path}'...")

# 1. Actual
df_actual_wide.to_csv(f"{path}/Result_Actual_EP.csv", encoding='cp949')

# 2. Pooled Models
df_ols_wide.to_csv(f"{path}/Result_Pred_Pooled_OLS.csv", encoding='cp949')
df_xgb_wide.to_csv(f"{path}/Result_Pred_Pooled_XGB.csv", encoding='cp949')

# 3. Individual Models
df_ols_wide_ind.to_csv(f"{path}/Result_Pred_Ind_OLS.csv", encoding='cp949')
df_xgb_wide_ind.to_csv(f"{path}/Result_Pred_Ind_XGB.csv", encoding='cp949')

print("Success! 모든 파일 저장이 완료되었습니다.")

---
---
## **이전 코드**

In [ ]:
# [현재 노트북의 마지막 셀]

import os

# 저장 경로 설정 (현재 폴더)
path = "."

print("Saving Results to CSV...")

# 1. OLS 예측값 (Wide Format: Date x Ticker)
df_ols_wide.to_csv(f"{path}/Result_Pred_OLS.csv", encoding='cp949')

# 2. XGBoost 예측값
df_xgb_wide.to_csv(f"{path}/Result_Pred_XGB.csv", encoding='cp949')

# 3. 실제 EP 값 (Actual)
df_actual_wide.to_csv(f"{path}/Result_Actual_EP.csv", encoding='cp949')

print("Success! 3개의 파일이 저장되었습니다.")
print("1. Result_Pred_OLS.csv")
print("2. Result_Pred_XGB.csv")
print("3. Result_Actual_EP.csv")

---
## **H1 검증**

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ---------------------------------------------------------
# H1 검증: 예측 성능 지표 (RMSE, MAE, R2_OOS)
# ---------------------------------------------------------
# 1. Error 계산 (전체 패널을 1줄로 펴서 계산)
# NaN은 제거 (데이터가 없는 구간 제외)
err_ols = (df_ols_wide - df_actual_wide).stack().dropna()
err_xgb = (df_xgb_wide - df_actual_wide).stack().dropna()

# 2. 성능 지표 계산 함수
def calc_metrics(errors):
    mse = np.mean(errors ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))
    return mse, rmse, mae

mse_ols, rmse_ols, mae_ols = calc_metrics(err_ols)
mse_xgb, rmse_xgb, mae_xgb = calc_metrics(err_xgb)

# 3. Out-of-Sample R-squared (Campbell & Thompson)
# R2_OOS = 1 - (MSE_Challenger / MSE_Benchmark)
# 양수(+)면 XGB가 더 잘한 것, 음수(-)면 OLS가 더 잘한 것
r2_oos = 1 - (mse_xgb / mse_ols)

# 4. 결과 리포트
h1_report = pd.DataFrame({
    'Metric': ['RMSE', 'MAE'],
    'Linear (OLS)': [rmse_ols, mae_ols],
    'XGBoost (ML)': [rmse_xgb, mae_xgb]
})

print("=== [H1 Result] Valuation Accuracy ===")
print(h1_report)
print("-" * 30)
print(f"Out-of-Sample R2 (ML vs Linear): {r2_oos*100:.4f}%")

if r2_oos > 0:
    print(">> Conclusion: 머신러닝이 선형 모델보다 예측 정확도가 높습니다. (H1 채택)")
else:
    print(">> Conclusion: 머신러닝의 예측력이 선형 모델보다 떨어집니다. (H1 기각 위기)")